# 03 Docker 容器化与部署

> 覆盖范围：容器 vs 虚拟机（**原理**）、镜像/容器/仓库三概念、Dockerfile 指令详解、常用命令、数据卷与网络、docker-compose、镜像构建缓存、K8s 入门衔接。
> 使用方式：先读**原理讲解**（为什么需要容器、镜像层如何复用），再看 **Dockerfile / 命令示例**（阅读与默写），最后用 Python 模拟核心语义验证，自测面试追问。

---

## 1. 为什么需要容器

### 1.1 部署的痛点

传统部署：每个环境（开发/测试/生产）手动装依赖——「在我机器上能跑」成为经典笑话。问题根源是**环境不一致**：

- 依赖版本冲突（A 要 Python 3.8，B 要 3.11）；
- 系统库/配置/路径差异；
- 多服务共享一台机器时的资源隔离。

### 1.2 容器 vs 虚拟机（本质区别）

| 维度 | 虚拟机 VM | 容器 Container |
| --- | --- | --- |
| 隔离层级 | **硬件级**（Hypervisor + 完整 Guest OS） | **进程级**（共享宿主内核） |
| 内核 | 每个 VM 一个 Guest OS | 共享宿主内核（Linux 内核） |
| 启动时间 | 分钟级（要启动完整 OS） | 秒级（启动进程） |
| 资源开销 | 大（每 VM 独占内存/CPU） | 小（只多一层进程隔离） |
| 镜像大小 | GB 级 | MB 级 |
| 隔离强度 | 强（完全独立内核） | 中（受限但共享内核） |

**容器的「隔离」依赖 Linux 内核特性**：

- **Namespace（命名空间）**：隔离视图——进程、网络、文件系统、PID、用户等各有一套命名空间，容器里看到的 `PID 1` 是容器自己的第一个进程，网络栈也是独立的；
- **Cgroups（控制组）**：限制资源——给容器设定 CPU、内存、IO 配额，防止一个容器吃光整机。

> **一句话**：VM 虚拟的是**硬件**，容器虚拟的是**操作系统**（进程视图 + 资源配额）。

## 2. 三大核心概念：镜像 / 容器 / 仓库

| 概念 | 本质 | 类比 |
| --- | --- | --- |
| **镜像 Image** | 只读模板（分层文件系统 + 启动命令 + 元数据） | 光盘 / 安装包 |
| **容器 Container** | 镜像的运行实例（可写层 + 进程） | 运行中的程序 |
| **仓库 Registry** | 存放/分发镜像（Docker Hub、私服 Harbor） | 应用商店 / Git 远端 |

**镜像分层（Union FS）**：镜像由多层只读层堆叠，底层不变就**复用、共享**：

- 拉取新镜像时只下载**新增的层**（Docker Hub 的层级缓存）；
- 同一个基础镜像（如 `python:3.11`）被多个镜像共享，磁盘只存一份；
- 容器运行时在上面加一层**可写层**（写时复制 Copy-on-Write）——容器删除时写层丢弃，镜像本身不变。

## 3. Dockerfile：镜像的构建脚本

Dockerfile 是**逐条指令生成镜像层**的声明式文件。核心指令：

| 指令 | 作用 | 注意 |
| --- | --- | --- |
| `FROM` | 基础镜像 | 必须第一行；选对基础镜像（尽量小：alpine / slim） |
| `WORKDIR` | 工作目录 | 后续 RUN/COPY/CMD 的相对路径基准；不存在会自动创建 |
| `COPY` / `ADD` | 拷贝文件进镜像 | ADD 可解压 tar/支持 URL（不常用）；COPY 更明确 |
| `RUN` | 构建期执行命令 | 每行 RUN 生成一层；**合并 RUN** 减少层数 |
| `ENV` | 环境变量 | 影响运行时 |
| `EXPOSE` | 声明端口 | 仅文档作用，真正映射靠 `docker run -p` |
| `CMD` | 默认启动命令 | 只有最后一个生效；可被 `docker run` 参数覆盖；适合做默认参数 |
| `ENTRYPOINT` | 固定入口 | 容器作为可执行程序时用；常与 CMD 配合（ENTRYPOINT 固定 + CMD 给默认参数） |
| `HEALTHCHECK` | 健康检查 | K8s 就绪探针的 Docker 侧来源 |

```dockerfile
# Python 服务示例
FROM python:3.11-slim          # 基础镜像（小体积）
WORKDIR /app                  # 工作目录
COPY requirements.txt .       # 先拷依赖清单（利用层缓存）
RUN pip install -r requirements.txt -i https://pypi.tuna.tsinghua.edu.cn/simple
COPY . .                      # 再拷贝源码（源码改动不重装依赖）
ENV PYTHONUNBUFFERED=1
EXPOSE 8000
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
```

**层缓存优化技巧**：把**变化慢的放前面**（依赖清单）、**变化快的放后面**（源码）。这样改一行代码不会触发 pip install 重跑。

## 4. 常用命令（阅读 + 默写）

```bash
# 镜像
docker build -t myapp:v1 .          # 构建镜像（-t 打标签）
docker images                      # 列出镜像
docker rmi myapp:v1                # 删除镜像

# 容器
docker run -d -p 8000:8000 --name app myapp:v1   # 后台运行，端口映射 宿主:容器
docker ps / docker ps -a           # 运行中 / 所有容器
docker exec -it app bash           # 进入容器 shell（-it 交互）
docker logs -f app                 # 跟随日志
docker stop / start / rm app       # 停止 / 启动 / 删除

# 数据卷（持久化）
docker run -v /host/path:/container/path myapp:v1   # 挂载宿主目录
docker volume ls

# 网络
docker network create mynet
docker run --network mynet --name svc1 myapp:v1

# 清理
docker system prune -a             # 清理所有未使用镜像/容器/缓存
```

**常见坑**：容器是**有状态的写层随容器删除而消失**——持久数据必须用 volume 挂载；容器内 `127.0.0.1` 是**容器自己**，不是宿主——跨容器访问用服务名/`--network`。

## 5. docker-compose：一键编排多容器

多个服务的典型配置（web + db + cache）：

```yaml
version: "3"
services:
  web:
    build: .
    ports: ["8000:8000"]
    depends_on: [db, redis]
  db:
    image: postgres:15
    environment:
      POSTGRES_PASSWORD: secret
    volumes: ["pgdata:/var/lib/postgresql/data"]
  redis:
    image: redis:7

volumes:
  pgdata:
```

```bash
docker compose up -d        # 启动全部
docker compose down         # 停止并清理
docker compose logs -f web  # 看某个服务日志
```

**与 K8s 的关系**：compose 面向**单机多容器**；Kubernetes 面向**多机集群**（Pod/Deployment/Service），概念上 compose 的 service ≈ k8s 的 Deployment+Service。


## 核心概念

### (a) 🗂️ 容器从「为什么」到「Compose 编排」的主线

这篇先讲容器解决了什么问题（环境不一致），再讲容器与虚拟机、Namespace/Cgroup 隔离原理，然后按「镜像 → 容器 → Dockerfile → 常用命令 → 卷与网络 → Compose → K8s」的顺序把容器化的完整链路串起来。

```mermaid
flowchart TB
    Node1["🧠 为什么需要容器：环境一致性"] --> Node2["🏠 容器 vs 虚拟机"]
    Node2 --> Node3["🪟 Namespace + Cgroup"]
    Node3 --> Node4["💿 镜像：只读分层模板"]
    Node4 --> Node5["📦 容器：镜像 + 可写层"]
    Node5 --> Node6["📜 Dockerfile：构建脚本"]
    Node6 --> Node7["🛠️ 常用命令：build / run / exec"]
    Node7 --> Node8["💾 数据卷 Volume + 网络"]
    Node8 --> Node9["🧩 docker-compose 多容器编排"]
    Node9 --> Node10["☸️ 衔接 K8s 集群"]
```

- **🧠 为什么需要容器**：部署的最大痛点是环境不一致——「在我机器上能跑」，依赖版本冲突、系统库差异、多服务资源共享。容器把应用连同依赖和配置一起打包，让开发/测试/生产环境完全一致。
- **🏠 容器 vs 虚拟机**：VM 虚拟的是**硬件**（每台一个完整 Guest OS，分钟级启动、GB 级镜像）；容器虚拟的是**操作系统**（共享宿主内核，只多一层进程隔离，秒级启动、MB 级镜像）。
- **🪟 Namespace + Cgroup**：容器的「隔离」靠两个 Linux 内核特性：Namespace 隔离视图（进程、网络、文件系统各有一套，容器里看到的是自己的 PID 1）；Cgroup 限制资源（CPU/内存/IO 配额），防止一个容器吃光整机。
- **💿 镜像**：只读的分层文件系统模板 + 启动命令 + 元数据，像光盘。镜像由多层只读层堆叠，底层不变就复用共享——同一个 `python:3.11` 基础镜像磁盘只存一份，拉取时只下载新增层。
- **📦 容器**：镜像是模板、容器是运行实例——运行时在镜像之上加一层可写层（写时复制 Copy-on-Write）。容器删除时写层跟着丢、镜像不变，所以持久数据必须挂卷，不能指望容器写层。
- **📜 Dockerfile**：逐条指令生成镜像层的构建脚本：`FROM` 定起点、`RUN` 装依赖、`COPY` 拷代码、`CMD`/`ENTRYPOINT` 定义启动。层缓存技巧：变化慢的放前面（requirements.txt）、变化快的放后面（源码），改一行代码不用重装依赖。
- **🛠️ 常用命令**：`docker build -t` 构建、`docker run -d -p 8000:8000` 后台运行 + 端口映射（宿主:容器）、`docker ps`/`logs`/`exec -it bash` 进容器调试、`system prune` 清理。
- **💾 数据卷 + 网络**：`-v 宿主目录:容器目录` 或具名 volume 做持久化；`--network` 让容器互通。记住：容器内 `127.0.0.1` 是容器自己不是宿主，跨容器访问用服务名。
- **🧩 docker-compose**：一个 yaml 描述 web + db + cache 等多个服务，`docker compose up -d` 一键启动全部；`depends_on` 控制启动顺序、`volumes` 声明数据卷。compose 面向单机多容器。
- **☸️ 衔接 K8s**：Kubernetes 把 compose 的单机编排升级成多机集群：Pod 是最小调度单元（一个或多个容器）、Deployment 管副本与滚动更新、Service 管服务发现——compose 的 service 概念上 ≈ K8s 的 Deployment + Service。


In [ ]:
# ---- matplotlib 中文字体（防图内中文乱码）----
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Songti SC', 'Arial Unicode MS', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
# -------------------------------------------
# 模拟 Docker 镜像分层构建 + 层缓存（理解 build 缓存原理）

class ImageLayer:
    """一个只读镜像层：记录其产生的文件变更"""
    def __init__(self, desc, files):
        self.desc = desc          # 如 'RUN pip install'
        self.files = files        # dict 文件名->内容

def build_image(instructions, cache):
    """按 Dockerfile 指令逐条构建，命中缓存层则跳过重建"""
    layers = []
    fs = {}
    for inst in instructions:
        # 缓存键：指令文本（简化为指令内容）
        if inst in cache:
            print(f'[cache hit] {inst}  <- 复用已有层，不重新执行')
            layers.append(cache[inst])
            fs.update(cache[inst].files)
        else:
            print(f'[build]     {inst}  <- 执行并生成新层')
            # 模拟执行效果：根据指令类型产出文件
            if inst.startswith('FROM'):
                new = {f'/base/{k}': v for k, v in {'py': '3.11'}.items()}
            elif inst.startswith('COPY requirements'):
                new = {'/app/requirements.txt': 'numpy==1.26\nflask==3.0'}
            elif inst.startswith('RUN pip'):
                new = {'/app/.venv': 'installed'}
            elif inst.startswith('COPY .'):
                new = {'/app/main.py': 'print(1)'}
            else:
                new = {}
            layer = ImageLayer(inst, new)
            layers.append(layer)
            cache[inst] = layer
            fs.update(new)
    return layers, fs

# 第一次构建：全部执行
cache = {}
dockerfile = [
    'FROM python:3.11-slim',
    'COPY requirements.txt .',
    'RUN pip install -r requirements.txt',
    'COPY . .',
]
print('=== 第一次构建 ===')
l1, fs1 = build_image(dockerfile, cache)
print(f'镜像层数: {len(l1)}, 最终文件: {sorted(fs1.keys())}\n')

# 第二次构建（只改了一行源码 -> Dockerfile 前三行不变）：命中缓存
print('=== 第二次构建（仅改 main.py）===')
l2, fs2 = build_image(dockerfile, cache)   # 理论上只需重建最后一步
print('层全部命中缓存（实际 Docker 中最后 COPY . . 层会因文件变化重建）')
assert len(l2) == 4

In [ ]:
# 模拟容器：镜像只读层 + 容器可写层（写时复制 Copy-on-Write）

class Container:
    def __init__(self, name, image_fs):
        self.name = name
        self.image_fs = image_fs       # 只读镜像层视图（共享）
        self.writable = {}             # 容器可写层（私有）

    def read(self, path):
        # 读：先查可写层，再查镜像层（层叠视图）
        if path in self.writable:
            return self.writable[path]
        return self.image_fs.get(path, 'NOT FOUND')

    def write(self, path, content):
        # 写：只写进可写层（镜像层不动 -> 镜像不可变）
        self.writable[path] = content

# 共享镜像
image = {'/app/main.py': 'v1.0', '/app/requirements.txt': 'numpy'}

c1 = Container('app-1', image)
c2 = Container('app-2', image)

print('c1 读镜像文件:', c1.read('/app/main.py'))
c1.write('/app/log.txt', 't1')          # 容器1写日志
c2.write('/app/log.txt', 't2')          # 容器2各自独立写
print('c1 写层:', c1.writable)
print('c2 写层:', c2.writable)
print('镜像层未被污染:', image)
print('删除 c1 后镜像仍存在（可再起新容器）')
assert '/app/log.txt' not in image, '可写层绝不能污染镜像层'

In [ ]:
# 模拟 Namespace 隔离 + Cgroup 资源限制（概念演示）

class FakeDockerDaemon:
    def __init__(self):
        self.containers = {}
        self.process_table = {}      # pid -> container_id（模拟 PID namespace）

    def run(self, cid, image, cmd, mem_limit_mb=None, cpu_quota=None):
        # 创建独立的命名空间视图
        ns = {
            'pid': {'1': cmd},                # 容器内 PID 1 = 容器的第一个进程
            'net': set(),                     # 独立网络栈（空 = 未映射端口）
            'mount': {'/'},                   # 独立文件系统挂载视图
            'uts': cid,                       # 独立主机名
        }
        self.containers[cid] = {'image': image, 'cmd': cmd, 'ns': ns,
                                'cgroup': {'mem_limit_mb': mem_limit_mb,
                                           'cpu_quota': cpu_quota}}
        return f'容器 {cid} 已启动：{cmd}（资源限制: 内存≤{mem_limit_mb}MB, CPU配额={cpu_quota}）'

    def exec_proc(self, cid, pid):
        ns = self.containers[cid]['ns']
        # 容器内进程看到的是命名空间内的视图
        return f'容器{cid} 内看到 PID 1: {ns["pid"]["1"]}，主机名: {ns["uts"]}'

d = FakeDockerDaemon()
print(d.run('c1', 'myapp:v1', 'python server.py', mem_limit_mb=512, cpu_quota=2))
print(d.run('c2', 'myapp:v1', 'python worker.py', mem_limit_mb=256, cpu_quota=1))
print(d.exec_proc('c1', 1))
print(d.exec_proc('c2', 1))
print('两个容器共享同一镜像，但进程/网络/主机名各自隔离（Namespace）')
print('内存/CPU 配额各自受限（Cgroup）')
assert d.containers['c1']['cgroup']['mem_limit_mb'] == 512

## 6. 面试 30 秒速答

- **容器 vs VM**：VM 虚拟硬件（完整 Guest OS），容器虚拟 OS（共享宿主内核，Namespace + Cgroup 隔离进程视图与资源）；
- **镜像分层**：只读层堆叠 + 底层共享；容器加可写层（写时复制）；镜像不可变；
- **镜像缓存**：按 Dockerfile 指令逐层缓存，变化慢的放前面（依赖）、快的放后面（源码）；
- **ENTRYPOINT vs CMD**：ENTRYPOINT 是固定入口（docker run 参数附加给它），CMD 是默认参数（可整体覆盖）；
- **数据持久化**：容器写层随容器删除；用 `-v` 挂载卷或具名 volume；
- **端口映射**：`-p 宿主:容器`；容器内 127.0.0.1 是容器自己；
- **compose vs k8s**：compose 单机编排；K8s 多机集群（Deployment/Pod/Service）。

## 7. 自测清单（面试追问）

- [ ] 容器与虚拟机的隔离原理有什么本质不同？（硬件虚拟化 vs Namespace/Cgroup）
- [ ] Namespace 与 Cgroup 各负责什么？（视图隔离 vs 资源限制）能列出几种 Namespace？
- [ ] 镜像为什么能共享底层？写时复制（CoW）是什么？容器删除后数据去哪了？
- [ ] Dockerfile 里 COPY 顺序为什么影响构建速度？（层缓存失效面）
- [ ] ENTRYPOINT 与 CMD 的区别？`docker run` 追加参数会覆盖谁？
- [ ] `docker run -d -p 8000:8000 app` 每个参数含义？容器如何访问宿主机？
- [ ] 容器内数据库数据会丢吗？怎么持久化？（volume 挂载）
- [ ] docker-compose 的 depends_on 解决什么问题？（启动顺序）
- [ ] 手写：一个 Python FastAPI 服务的 Dockerfile；一条 compose 配置（web+db）；
- [ ] 基础镜像为什么用 slim/alpine？（体积 + 攻击面）；多阶段构建是什么？
- [ ] K8s 的 Pod 与 Deployment 是什么？和 docker 容器什么关系？

> ✅ 全部能回答 + 能默写代码，本课通过。